In [0]:
%sql
SELECT * FROM soccer.carlosalbertomurillomartinez.player_stats LIMIT 10

#### Read job parameters

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target_schema = f"{catalog}.silver"
print(f"Catalog: {catalog}, Schema: {schema}")

In [0]:
from pyspark.sql.functions import col, when, get_json_object, inline
df = spark.read.table(f'{catalog}.{schema}.player_stats')


df_normalized_player_stats = (
    df
    .withColumn('games_played', col('games').getItem('appearences'))
    .withColumn('games_is_captain', col('games').getItem('captain'))
    .withColumn('games_minutes', col('games').getItem('minutes'))
    .withColumn('games_yellow_cards', col('cards').getItem('yellow'))
    .withColumn('games_yellow_red_cards', col('cards').getItem('yellowred'))
    .withColumn('games_red_cards', col('cards').getItem('red'))
    .withColumn('goals_total', col('goals').getItem('total'))
    .withColumn('goals_assists', col('goals').getItem('assists'))
    .withColumn('goals_saves', col('goals').getItem('saves'))
    .withColumn('passes_total', col('passes').getItem('total'))
    .withColumn('passes_key', col('passes').getItem('key'))
    .withColumn('passes_accuracy', col('passes').getItem('accuracy'))
    .withColumn('penalty_won', col('penalty').getItem('won'))
    .withColumn('penalty_committed', col('penalty').getItem('commited'))
    .withColumn('penalty_scored', col('penalty').getItem('scored'))
    .withColumn('penalty_missed', col('penalty').getItem('missed'))
    .withColumn('penalty_saved', col('penalty').getItem('saved'))
    .withColumn('dribbles_attempts', col('dribbles').getItem('attempts'))
    .withColumn('dribbles_success', col('dribbles').getItem('success'))
    .withColumn('dribbles_past', col('dribbles').getItem('past'))
    .withColumn('fouls_drawn', col('fouls').getItem('drawn'))
    .withColumn('fouls_committed', col('fouls').getItem('committed'))
    .withColumn('age_rescued', get_json_object(col('_rescued_data'), '$.age'))
    .withColumn('age', when(col('age_rescued').isNotNull(), col('age_rescued')).otherwise(col('age')))
    .select(
        'player_id', 'league', 'season', 'age',
        'games_played', 'games_is_captain', 'games_minutes',
        'games_yellow_cards', 'games_yellow_red_cards', 'games_red_cards',
        'goals_total', 'goals_assists', 'goals_saves',
        'passes_total', 'passes_key', 'passes_accuracy',
        'penalty_won', 'penalty_committed', 'penalty_scored', 'penalty_missed', 'penalty_saved',
        'dribbles_attempts', 'dribbles_success', 'dribbles_past',
        'fouls_drawn', 'fouls_committed'
    )
)


obtain player data on a column format generate a new dataframe that will be used for analysis

In [0]:
df = spark.read.table('soccer.carlosalbertomurillomartinez.teams_statistics')
display(df.limit(10).toPandas())

In [0]:
# Flatten all struct columns into individual columns
df_normalized = (
    df
    # Biggest losses
    .withColumn('biggest_losses_home', col('biggest_losses').getItem('home'))
    .withColumn('biggest_losses_away', col('biggest_losses').getItem('away'))
    # Biggest wins
    .withColumn('biggest_wins_home', col('biggest_wins').getItem('home'))
    .withColumn('biggest_wins_away', col('biggest_wins').getItem('away'))
    # Fixtures draws
    .withColumn('fixtures_draws_home', col('fixtures_draws').getItem('home'))
    .withColumn('fixtures_draws_away', col('fixtures_draws').getItem('away'))
    .withColumn('fixtures_draws_total', col('fixtures_draws').getItem('total'))
    # Fixtures losses
    .withColumn('fixtures_losses_home', col('fixtures_losses').getItem('home'))
    .withColumn('fixtures_losses_away', col('fixtures_losses').getItem('away'))
    .withColumn('fixtures_losses_total', col('fixtures_losses').getItem('total'))
    # Fixtures wins
    .withColumn('fixtures_wins_home', col('fixtures_wins').getItem('home'))
    .withColumn('fixtures_wins_away', col('fixtures_wins').getItem('away'))
    .withColumn('fixtures_wins_total', col('fixtures_wins').getItem('total'))
    # Fixtures played
    .withColumn('fixtures_played_home', col('fixtures_played').getItem('home'))
    .withColumn('fixtures_played_away', col('fixtures_played').getItem('away'))
    .withColumn('fixtures_played_total', col('fixtures_played').getItem('total'))
    # Goals against - averages
    .withColumn('goals_against_avg_home', col('goals_against').getItem('average').getItem('home'))
    .withColumn('goals_against_avg_away', col('goals_against').getItem('average').getItem('away'))
    .withColumn('goals_against_avg_total', col('goals_against').getItem('average').getItem('total'))
    # Goals against - totals
    .withColumn('goals_against_total_home', col('goals_against').getItem('total').getItem('home'))
    .withColumn('goals_against_total_away', col('goals_against').getItem('total').getItem('away'))
    .withColumn('goals_against_total_total', col('goals_against').getItem('total').getItem('total'))
    # Goals for - averages
    .withColumn('goals_for_avg_home', col('goals_for').getItem('average').getItem('home'))
    .withColumn('goals_for_avg_away', col('goals_for').getItem('average').getItem('away'))
    .withColumn('goals_for_avg_total', col('goals_for').getItem('average').getItem('total'))
    # Goals for - totals
    .withColumn('goals_for_total_home', col('goals_for').getItem('total').getItem('home'))
    .withColumn('goals_for_total_away', col('goals_for').getItem('total').getItem('away'))
    .withColumn('goals_for_total_total', col('goals_for').getItem('total').getItem('total'))
    # Penalties
    .withColumn('penalties_missed_pct', col('penalties_missed').getItem('percentage'))
    .withColumn('penalties_missed_total', col('penalties_missed').getItem('total'))
    .withColumn('penalties_scored_pct', col('penalties_scored').getItem('percentage'))
    .withColumn('penalties_scored_total', col('penalties_scored').getItem('total'))
    # Red cards by time period
    .withColumn('red_cards_0_15_total', col('red_cards').getItem('0-15').getItem('total'))
    .withColumn('red_cards_16_30_total', col('red_cards').getItem('16-30').getItem('total'))
    .withColumn('red_cards_31_45_total', col('red_cards').getItem('31-45').getItem('total'))
    .withColumn('red_cards_46_60_total', col('red_cards').getItem('46-60').getItem('total'))
    .withColumn('red_cards_61_75_total', col('red_cards').getItem('61-75').getItem('total'))
    .withColumn('red_cards_76_90_total', col('red_cards').getItem('76-90').getItem('total'))
    .withColumn('red_cards_91_105_total', col('red_cards').getItem('91-105').getItem('total'))
    .withColumn('red_cards_106_120_total', col('red_cards').getItem('106-120').getItem('total'))
    # Yellow cards by time period
    .withColumn('yellow_cards_0_15_total', col('yellow_cards').getItem('0-15').getItem('total'))
    .withColumn('yellow_cards_16_30_total', col('yellow_cards').getItem('16-30').getItem('total'))
    .withColumn('yellow_cards_31_45_total', col('yellow_cards').getItem('31-45').getItem('total'))
    .withColumn('yellow_cards_46_60_total', col('yellow_cards').getItem('46-60').getItem('total'))
    .withColumn('yellow_cards_61_75_total', col('yellow_cards').getItem('61-75').getItem('total'))
    .withColumn('yellow_cards_76_90_total', col('yellow_cards').getItem('76-90').getItem('total'))
    .withColumn('yellow_cards_91_105_total', col('yellow_cards').getItem('91-105').getItem('total'))
    .withColumn('yellow_cards_106_120_total', col('yellow_cards').getItem('106-120').getItem('total'))
    # Select flattened columns
    .select(
        'team_id', 'league', 'season', 'name',
        'biggest_losses_home', 'biggest_losses_away',
        'biggest_wins_home', 'biggest_wins_away',
        'clean_sheets_home', 'clean_sheets_away', 'clean_sheets_total',
        'failed_to_score_home', 'failed_to_score_away', 'failed_to_score_total',
        'fixtures_draws_home', 'fixtures_draws_away', 'fixtures_draws_total',
        'fixtures_losses_home', 'fixtures_losses_away', 'fixtures_losses_total',
        'fixtures_wins_home', 'fixtures_wins_away', 'fixtures_wins_total',
        'fixtures_played_home', 'fixtures_played_away', 'fixtures_played_total',
        'goals_against_avg_home', 'goals_against_avg_away', 'goals_against_avg_total',
        'goals_against_total_home', 'goals_against_total_away', 'goals_against_total_total',
        'goals_for_avg_home', 'goals_for_avg_away', 'goals_for_avg_total',
        'goals_for_total_home', 'goals_for_total_away', 'goals_for_total_total',
        'penalties_missed_pct', 'penalties_missed_total',
        'penalties_scored_pct', 'penalties_scored_total',
        'red_cards_0_15_total', 'red_cards_16_30_total', 'red_cards_31_45_total',
        'red_cards_46_60_total', 'red_cards_61_75_total', 'red_cards_76_90_total',
        'red_cards_91_105_total', 'red_cards_106_120_total',
        'yellow_cards_0_15_total', 'yellow_cards_16_30_total', 'yellow_cards_31_45_total',
        'yellow_cards_46_60_total', 'yellow_cards_61_75_total', 'yellow_cards_76_90_total',
        'yellow_cards_91_105_total', 'yellow_cards_106_120_total',
        'form'
    )
)


In [0]:
df = spark.read.table(f'{catalog}.{schema}.teams_statistics')

df_formations = (
    df
    .select(
        col('league').alias('league_id'),
        col('season'),
        col('team_id'),
        inline('lineups')
    )
)


In [0]:
silver_schema = f"{catalog}.silver"
spark.sql(f"create database if not exists {silver_schema}")
df_normalized.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.teams_statistics")
df_formations.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.teams_formations")
df_normalized_player_stats.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.player_stats")